# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a Croissant-described dataset using the `mlcroissant` library.

The workflow follows a simple pattern:
- load the dataset metadata from a Croissant JSON-LD URL,
- inspect the available record sets and fields,
- load selected record sets into pandas DataFrames,
- perform basic exploratory data analysis (EDA), and
- create a few example visualizations.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Important note on identifiers
In Croissant, entities such as record sets and fields are uniquely identified by their `@id`. In this notebook, we always reference record sets and fields by their `@id` values when interacting with the dataset.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the Croissant metadata and create a dataset object with `mlcroissant`.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))
print('Citation:', getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', 'N/A')))

We can also inspect the metadata in JSON form. This is useful for discovering the dataset structure, including record sets and fields.

In [ ]:
metadata_json = metadata.to_json()

print(type(metadata_json))
print('\nTop-level metadata keys:')
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review the available record sets, fields, and their `@id` values.

Because Croissant datasets may expose many tables, this step helps identify which record sets are actually present and which fields are useful for analysis.

In [ ]:
# Discover record sets from the metadata JSON
record_sets_json = metadata_json.get('recordSet', [])

print(f'Number of record sets declared in metadata JSON: {len(record_sets_json)}')

record_set_rows = []
for rs in record_sets_json:
    rs_id = rs.get('@id', '')
    rs_name = rs.get('name', '')
    rs_description = rs.get('description', '')
    field_count = len(rs.get('field', [])) if isinstance(rs.get('field', []), list) else 0
    record_set_rows.append({
        'record_set_id': rs_id,
        'record_set_name': rs_name,
        'field_count': field_count,
        'description': rs_description
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

If record sets are present, the next cell expands the field-level structure. Every field is also tracked by its `@id`.

In [ ]:
field_rows = []

for rs in record_sets_json:
    rs_id = rs.get('@id', '')
    rs_name = rs.get('name', '')
    for field in rs.get('field', []) if isinstance(rs.get('field', []), list) else []:
        source_info = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field.get('@id', ''),
            'field_name': field.get('name', ''),
            'data_type': field.get('dataType', ''),
            'description': field.get('description', ''),
            'source_file': source_info.get('fileObject', ''),
            'source_extract': source_info.get('extract', ''),
            'source_field': source_info.get('field', ''),
            'column': field.get('column', '')
        })

fields_df = pd.DataFrame(field_rows)
print(f'Number of fields discovered: {len(fields_df)}')
fields_df.head(20)

The Croissant package summary supplied with this dataset indicates a rich provenance-oriented dataset, but the live metadata may or may not expose record sets through the `recordSet` section at runtime.

The next cell checks whether record sets are directly accessible for loading.

In [ ]:
available_record_set_ids = record_sets_df['record_set_id'].tolist() if not record_sets_df.empty else []
print('Available record set @id values:')
print(available_record_set_ids if available_record_set_ids else 'No record sets found in metadata JSON.')

When a record set is available, `dataset.records(record_set=<record_set_id>)` returns the rows for that table. The next example is written defensively so the notebook still runs if the public Croissant description does not expose record sets.

In [ ]:
if available_record_set_ids:
    example_record_set_id = available_record_set_ids[0]
    print('Example record set @id:', example_record_set_id)
    
    try:
        for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
            print(record)
            if i >= 2:
                break
    except Exception as exc:
        print('Unable to stream records from the example record set.')
        print(type(exc).__name__, str(exc))
else:
    print('No record set @id is available to demonstrate dataset.records(...).')

## 3. Data Extraction
Load one or more record sets into pandas DataFrames.

This section uses variables to dynamically handle record set IDs. If the Croissant metadata currently exposes no record sets, the notebook will skip loading gracefully.

In [ ]:
# Select record sets dynamically from discovered metadata
record_set_ids = available_record_set_ids[:3]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(dataframes[record_set_id])} rows from record set: {record_set_id}')
    except Exception as exc:
        print(f'Could not load record set {record_set_id}')
        print(type(exc).__name__, str(exc))

print('\nLoaded DataFrame keys:')
print(list(dataframes.keys()))

In [ ]:
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print('First loaded record set @id:', first_record_set_id)
    print('Columns:')
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record sets were loaded into DataFrames.')

To make downstream analysis easier, the next helper cell summarizes each loaded table.

In [ ]:
table_summary_rows = []

for record_set_id, df in dataframes.items():
    table_summary_rows.append({
        'record_set_id': record_set_id,
        'n_rows': len(df),
        'n_columns': len(df.columns),
        'columns': ', '.join(df.columns.astype(str).tolist()[:15])
    })

table_summary_df = pd.DataFrame(table_summary_rows)
table_summary_df if not table_summary_df.empty else pd.DataFrame({'message': ['No tables loaded']})

## 4. Exploratory Data Analysis (EDA)
This section performs generic EDA that can adapt to whichever record sets are available.

The examples below show how to:
- identify numeric and categorical fields,
- filter rows based on a numeric threshold,
- normalize a numeric field,
- group by a categorical field, and
- inspect missingness.

All operations use the loaded record set and field names dynamically.

In [ ]:
analysis_record_set_id = None
analysis_df = None

for record_set_id, df in dataframes.items():
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    if numeric_columns:
        analysis_record_set_id = record_set_id
        analysis_df = df.copy()
        break

if analysis_df is not None:
    print('Selected record set for analysis:', analysis_record_set_id)
    print('Shape:', analysis_df.shape)
    print('\nNumeric columns:')
    print(analysis_df.select_dtypes(include='number').columns.tolist())
    print('\nNon-numeric columns:')
    print(analysis_df.select_dtypes(exclude='number').columns.tolist()[:30])
else:
    print('No loaded record set contains numeric columns suitable for this EDA example.')

In [ ]:
if analysis_df is not None:
    numeric_field_id = analysis_df.select_dtypes(include='number').columns.tolist()[0]
    numeric_series = analysis_df[numeric_field_id].dropna()
    threshold = numeric_series.quantile(0.75) if not numeric_series.empty else 0

    filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()
    print(f'Using numeric field: {numeric_field_id}')
    print(f'Threshold (75th percentile): {threshold}')
    print(f'Filtered rows: {len(filtered_df)}')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field_id].std()
    if len(filtered_df) > 1 and pd.notna(std_value) and std_value != 0:
        normalized_field_id = f'{numeric_field_id}_normalized'
        filtered_df[normalized_field_id] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        display(filtered_df[[numeric_field_id, normalized_field_id]].head())
    else:
        print('Normalization skipped because the filtered sample is too small or has zero variance.')
else:
    print('Filtering and normalization skipped.')

In [ ]:
if analysis_df is not None:
    categorical_candidates = [
        col for col in analysis_df.columns
        if analysis_df[col].dtype == 'object' or str(analysis_df[col].dtype).startswith('string')
    ]
    
    group_field_id = None
    for col in categorical_candidates:
        nunique = analysis_df[col].nunique(dropna=True)
        if 1 < nunique <= 20:
            group_field_id = col
            break
    
    if group_field_id is not None:
        print('Grouping field selected:', group_field_id)
        grouped_df = analysis_df.groupby(group_field_id).mean(numeric_only=True)
        display(grouped_df.head(10))
    else:
        print('No suitable categorical grouping field was found.')
else:
    print('Grouping step skipped.')

In [ ]:
if analysis_df is not None:
    missingness_df = (
        analysis_df.isna()
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    missingness_df.columns = ['field_id', 'missing_fraction']
    display(missingness_df.head(20))
else:
    print('Missingness analysis skipped.')

### Optional metadata-focused exploration
Even if record sets are not available, the metadata itself can still be explored. The next cell summarizes key descriptive attributes from the dataset package.

In [ ]:
metadata_summary = pd.DataFrame([
    {'attribute': 'name', 'value': getattr(metadata, 'name', 'N/A')},
    {'attribute': 'version', 'value': getattr(metadata, 'version', 'N/A')},
    {'attribute': 'license', 'value': getattr(metadata, 'license', 'N/A')},
    {'attribute': 'url', 'value': croissant_url},
    {'attribute': 'description', 'value': getattr(metadata, 'description', 'N/A')}
])

metadata_summary

## 5. Visualization
Visualize distributions or simple summary patterns from the loaded data.

The code below adapts to the available data. If no loadable numeric record set exists, it falls back to a metadata-based visualization.

In [ ]:
if analysis_df is not None:
    numeric_field_id = analysis_df.select_dtypes(include='number').columns.tolist()[0]
    
    plt.figure(figsize=(8, 4))
    sns.histplot(analysis_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of numeric field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Numeric histogram skipped because no numeric analysis table is available.')

In [ ]:
if analysis_df is not None:
    numeric_cols = analysis_df.select_dtypes(include='number').columns.tolist()
    if len(numeric_cols) >= 2:
        x_field_id = numeric_cols[0]
        y_field_id = numeric_cols[1]
        
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=analysis_df, x=x_field_id, y=y_field_id)
        plt.title(f'{y_field_id} vs {x_field_id}')
        plt.tight_layout()
        plt.show()
    else:
        print('Scatter plot skipped because fewer than two numeric fields are available.')
else:
    print('Scatter plot skipped.')

In [ ]:
if not record_sets_df.empty:
    plot_df = record_sets_df.sort_values('field_count', ascending=False).head(15)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='field_count', y='record_set_id', palette='viridis')
    plt.title('Field counts by record set @id')
    plt.xlabel('Number of fields')
    plt.ylabel('Record set @id')
    plt.tight_layout()
    plt.show()
else:
    fallback_df = pd.DataFrame({
        'metric': ['authors', 'citations', 'distributions', 'record_sets'],
        'count': [
            len(metadata_json.get('author', [])),
            len(metadata_json.get('citation', [])),
            len(metadata_json.get('distribution', [])),
            len(metadata_json.get('recordSet', []))
        ]
    })
    
    plt.figure(figsize=(7, 4))
    sns.barplot(data=fallback_df, x='metric', y='count', palette='magma')
    plt.title('Dataset package summary from metadata')
    plt.xlabel('Metadata component')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook showed how to use `mlcroissant` to inspect a Croissant dataset and work with its metadata and record sets.

### Key takeaways
- The dataset can be initialized directly from its Croissant JSON-LD URL.
- Metadata inspection is the first step in discovering the structure of a Croissant dataset.
- Record sets and fields should be referenced by their `@id` values.
- Once record sets are loaded into pandas DataFrames, standard EDA workflows can be applied.
- Defensive, metadata-driven code is useful because some published Croissant packages may expose rich descriptive metadata even when record sets are not immediately available in the `recordSet` section.

### Next steps
You can extend this notebook by:
- selecting specific record set `@id` values once they are confirmed in the metadata,
- joining related record sets using identifier fields,
- validating provenance and review-state columns,
- filtering synthetic versus empirical records where relevant, and
- building domain-specific analyses for food, nutrient, hazard, phenotype, or provenance tables.